# Natural Language Processing Coursework

## Setup

In [15]:


import os
if not os.path.exists("NLPLabs-2024"):
    !git clone -q https://github.com/CRLala/NLPLabs-2024.git


if not os.path.exists("dontpatronizeme"):
    !git clone -q https://github.com/Perez-AlmendrosC/dontpatronizeme.git

    
pcl_tsv_path = "NLPLabs-2024/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
train_split_path = "dontpatronizeme/semeval-2022/practice splits/train_semeval_parids-labels.csv"
dev_split_path   = "dontpatronizeme/semeval-2022/practice splits/dev_semeval_parids-labels.csv"


In [3]:
import os
from pathlib import Path
import re
import csv
import pandas as pd
import matplotlib.pyplot as plt

In [20]:

rows = []
with open(pcl_tsv_path, "r", encoding="utf-8", newline="") as f:
    r = csv.reader(f, delimiter="\t")
    for parts in r:
        if not parts:
            continue

        # skip boilerplate/disclaimer lines
        if len(parts) == 1:
            continue

        # expect 6 columns for real data
        if len(parts) != 6:
            raise ValueError(f"Unexpected number of columns: {len(parts)} | {parts[:3]}")

        pid, article_id, keyword, country, paragraph, label = parts
        rows.append({
            "paragraph_id": str(pid),
            "article_id": str(article_id),
            "keyword": str(keyword),
            "country_code": str(country),
            "paragraph": str(paragraph),
            "label_orig": int(label),
            "label_binary": 1 if int(label) >= 2 else 0,
        })

df = pd.DataFrame(rows)
df.head()

,paragraph_id,article_id,keyword,country_code,paragraph,label_orig,label_binary
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0
2,3,@@16584954,immigrant,ie,White House press secretary Sean Spicer said t...,0,0
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0
4,5,@@1494111,refugee,ca,""" Just like we received migrants fleeing El Sa...",0,0


In [10]:
train_split = pd.read_csv(train_split_path)
dev_split   = pd.read_csv(dev_split_path)

train_ids = set(train_split.iloc[:, 0].astype(str))
dev_ids   = set(dev_split.iloc[:, 0].astype(str))

df_train = df[df["paragraph_id"].isin(train_ids)].reset_index(drop=True)
df_dev   = df[df["paragraph_id"].isin(dev_ids)].reset_index(drop=True)

print("All:", len(df), "Train:", len(df_train), "Dev:", len(df_dev))

All: 10469 Train: 8375 Dev: 2094


## 2. EDA

### 2.1. Basic Statistical Profiling